# ML + NLP Challenge — Predicting the political orientation of news articles

**Team members:** _name 1_, _name 2_

## Goal
Build an end-to-end pipeline (text preprocessing → features → classifier → predictions) that predicts
the political orientation of a news article: `left`, `lean left`, `center`, `lean right`, `right`.

## Data
| File | Rows | Labels | Use |
|---|---|---|---|
| `data/train_dataset.csv` | 5,600 | yes (`Bias`) | everything you need to build and evaluate your models |
| `data/test_dataset.csv` | 2,183 | **no** | add your predictions in a `Bias` column and submit it as `test_dataset_predicted.csv` |

Columns: `id`, `Title`, `Text`, `Bias`. The articles were **scraped from the web**.

Both files are **raw**: they have **not been cleaned**. Inspecting and cleaning them is part of the challenge.

You must return a prediction for **every** row of `test_dataset.csv`. Test rows that contain no usable
article are not graded.

## Rules
- Your final score is computed by the instructor on the hidden labels of `test_dataset.csv`. Main metric: **macro-F1**; accuracy is also reported.
- `test_dataset.csv` has no labels: to compare models, **split `train_dataset.csv` yourselves** (e.g. train / validation).
- The notebook must run top to bottom (*Restart & Run All*) without manual edits.

## Submission
On **Lola**, **one submission per group**, by **25 November 2026, 23:59**. Upload two files:
1. This notebook (`.ipynb`), completed and runnable top to bottom.
2. `data/test_dataset_predicted.csv`: `test_dataset.csv` with an added `Bias` column holding your prediction for every row (written by section 7).

## 0. Setup
Fix the random seed and record the package versions so that your results can be reproduced.

In [ ]:
# 0. Setup
# !pip install -q pandas scikit-learn    # uncomment if needed

import random
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
# If you use PyTorch / transformers, also call torch.manual_seed(SEED)

DATA_DIR = Path("data")
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

## 1. Data loading

In [ ]:
# 1. Data loading
train_df = pd.read_csv(DATA_DIR / "train_dataset.csv")
test_df = pd.read_csv(DATA_DIR / "test_dataset.csv")
print("train:", train_df.shape, "| test:", test_df.shape)

In [ ]:
train_df.head()

## 2. Exploration
Before modelling, look at the data. Some questions to guide you:
- How are the labels distributed? What does that imply for the choice of metric?
- Are there missing values? Empty or suspicious texts? Duplicates?
- Do the texts and titles read correctly? Look at several random examples.
- How long are the articles? Are some suspiciously short?
- Look at `test_dataset.csv` too: does it have the same problems?

In [ ]:
# 2. Exploration
train_df["Bias"].value_counts()

In [ ]:
# TODO: explore the data (missing values, duplicates, text length, random examples, ...)

## 3. Cleaning and preprocessing
Two different steps. For each one, explain **why** it is needed and show its effect (e.g. rows before / after).

### 3.1 Cleaning
Fix or remove what is broken, based on what you found in the exploration. Some questions to guide you:
- Which characters do not display correctly? Can they be repaired rather than deleted?
- Which rows contain no usable article? Should they stay in the training data?
- Are some articles present several times? With the same label?

⚠️ Two kinds of cleaning, two rules:
- **Repairing the text** (e.g. fixing characters): apply the same function to `train_dataset.csv` **and** `test_dataset.csv`.
- **Removing rows**: only in `train_dataset.csv`. The test set must keep **every** row, since you predict all of them.

In [ ]:
# 3.1 Cleaning
# TODO: text repairs -> train_df and test_df | row removals -> train_df only

### 3.2 Text preprocessing (train **and** test)
Transformations that prepare the text for the model, e.g. combining `Title` and `Text`, lower-casing,
removing punctuation, numbers or stop words, stemming / lemmatisation.
Whether each step helps is an empirical question: compare on your validation set.

⚠️ Apply **exactly the same** preprocessing function to the training texts and to the texts of `test_dataset.csv`,
and predict **every** test row. Putting the preprocessing inside a `sklearn` `Pipeline` guarantees this.

In [ ]:
# 3.2 Text preprocessing
# TODO: write a preprocessing function and apply it to both train_df and test_df

## 4. Train / validation split
`test_dataset.csv` has no labels, so you need your own validation set to compare models.
Do it **after** cleaning. Think about how to split: stratification? Could the same article end up on both sides of the split?

In [ ]:
# 4. Train / validation split
from sklearn.model_selection import train_test_split

train_part, val_part = train_test_split(
    train_df, test_size=0.2, stratify=train_df["Bias"], random_state=SEED
)
print("train:", train_part.shape, "| validation:", val_part.shape)

## 5. Baseline: TF-IDF + Logistic Regression on the raw text
A simple reference model, with **no cleaning**. Your goal is to beat it, and to explain why your model does better.

In [ ]:
# 5. Baseline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import Pipeline

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20_000)),
    ("clf", LogisticRegression(max_iter=1000)),
])
baseline.fit(train_part["Text"].fillna(""), train_part["Bias"])

val_pred = baseline.predict(val_part["Text"].fillna(""))
print(f"Baseline | accuracy = {accuracy_score(val_part['Bias'], val_pred):.3f}"
      f" | macro-F1 = {f1_score(val_part['Bias'], val_pred, average='macro'):.3f}")
print(classification_report(val_part["Bias"], val_pred, zero_division=0))

## 6. Your models
Improve on the baseline: cleaning, features (n-grams, title, embeddings, ...), models
(linear models, tree ensembles, fine-tuned transformers, ...), class imbalance, hyper-parameter tuning.

Keep a short record of what you tried and the validation score of each attempt.

| Model | Validation accuracy | Validation macro-F1 | Notes |
|---|---|---|---|
| Baseline (TF-IDF + LR, raw text) | | | |
| | | | |

In [ ]:
# 6. Your models
# TODO

## 7. Final model and predictions
Train your final model (you may refit it on all of `train_dataset.csv`), save it, and predict **every** row of `test_dataset.csv`.

In [ ]:
# 7. Final model and predictions
import pickle

final_model = baseline   # TODO: replace with your best model
final_model.fit(train_df["Text"].fillna(""), train_df["Bias"])

with open("model_v1.pkl", "wb") as f:
    pickle.dump(final_model, f)

test_predicted = test_df.copy()
test_predicted["Bias"] = final_model.predict(test_df["Text"].fillna(""))

In [ ]:
# Check the submission format before saving
LABELS = {"left", "lean left", "center", "lean right", "right"}
assert len(test_predicted) == len(test_df) and set(test_predicted["id"]) == set(test_df["id"])
assert test_predicted["Bias"].isin(LABELS).all()

# Save under a new name: never overwrite the input file
test_predicted.to_csv(DATA_DIR / "test_dataset_predicted.csv", index=False)
test_predicted["Bias"].value_counts()

## 8. Critical thinking
Write a short discussion (a few paragraphs). Some questions to consider:
- How were the labels produced? What exactly does a label describe — the article, or something else?
- What might your model actually be learning? How could you check?
- Who could misuse a "political bias detector", and how?
- What are the limits of this dataset (period covered, outlets, scraping quality)?

_Your answer here._

## 9. Conclusion
Summarise your final model, its validation score, what worked, what did not, and what you would try next.

_Your answer here._